# USGS Landsat — Complete Data Access & Processing Tutorial

This notebook walks through the **entire USGS Landsat workflow** in
PyGeoFetch: authentication, search, download, radiometric scaling, cloud
masking, spectral indices, and visualization — using real Landsat
Collection 2 Level-2 data.

| Stage | Method | Reference |
|---|---|---|
| Authentication | M2M Application Token (`login-token`) | USGS M2M Application Token Documentation |
| Search | Landsat 8/9 Collection 2 Level-2 | USGS M2M API |
| Download | Dynamic product resolution + resumable streaming | USGS `download-options` / `download-request` |
| Radiometric scaling | DN → surface reflectance / temperature | USGS Landsat C2L2 Science Product Guide |
| Cloud masking | QA_PIXEL bit-flag decoding | Landsat 8-9 C2 Level 2 Science Product Guide, Table 6-2/6-3 |
| Spectral indices | NDVI, NDWI, NDBI, NBR | `pygeofetch.processor.SpectralIndex` |

## Before you start — two USGS-specific prerequisites

**1. You need an M2M Application Token, not your ERS password.**
USGS deprecated password-based login on 2025-02-26. Generate a token at:
`https://ers.cr.usgs.gov` → profile → "Application Token"

**2. Your account needs M2M access APPROVED — a separate manual step.**
Even with a valid token, API calls fail until USGS approves M2M access
for your account. Request it at:
`https://ers.cr.usgs.gov/profile/access`
This is a manual review process — approval is not instant, budget a few
days if you haven't requested this before.


In [1]:
# Verify the environment
import importlib.util

for pkg in ["rasterio", "numpy", "matplotlib"]:
    print(f"  [{'OK' if importlib.util.find_spec(pkg) else 'MISSING'}] {pkg}")


  [OK] rasterio
  [OK] numpy
  [OK] matplotlib


## Part 1 — Authentication

Pass your Application Token via `api_key` — **not** `password`.


In [ ]:
from pathlib import Path
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions

client = PyGeoFetch(log_level="INFO")

Replace with your own ERS username and Application Token.
DO NOT put your ERS account password here — that login method is
deprecated and will not work.
USGS_USERNAME = "your_ers_username"
USGS_APP_TOKEN = "your_application_token"

try:
    client.add_credentials("usgs", username=USGS_USERNAME, api_key=USGS_APP_TOKEN)
    print("Credentials registered. Authentication happens on first search/download call.")
except Exception as exc:
    print(f"Could not register credentials: {exc}")


12:56:14 INFO [      engine] PyGeoFetch ready


## Part 2 — Search Landsat Collection 2 Level-2

We search over a small area of interest for recent, low-cloud Landsat 8/9
scenes. Collection 2 Level-2 products are already atmospherically
corrected (surface reflectance) — the right starting point for most
vegetation/water/urban analyses, since you don't need to run your own
atmospheric correction.


In [ ]:
# Example AOI — adjust freely. This one covers a small area near Accra, Ghana.
aoi = BoundingBox.from_string("-0.30,5.55,-0.10,5.75")

query = SearchQuery(
    bbox=aoi,
    start_date="2024-01-01",
    end_date="2024-12-31",
    satellites=["Landsat-8", "Landsat-9"],
    cloud_cover_max=15,
    max_results=10,
)

try:
    results = client.search(query, providers=["usgs"])
except Exception as exc:
    print(f"Search failed: {exc}")
    results = []

print(f"Found {len(results)} Landsat scenes:\n")
for r in results:
    print(f"  {r.id}  |  {r.datetime}  |  cloud: {r.cloud_cover}%")


┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : planetary_computer                                          │
│ BBox       : [-0.300, 5.550, -0.100, 5.750]                              │
│ Date range : 2024-01-01  →  2024-12-31                                   │
│ Cloud max  : 15.0%                                                       │
│ Product    : any                                                         │
└──────────────────────────────────────────────────────────────────────────┘
12:56:19 INFO [planetary_computer] Planetary Computer: 7 results
  ✓  planetary_computer               7 scenes   1.4s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │    DATE    │   SATELLITE    │ CLOUD  │ PRODUCT │ POLARISATION │    PASS     │ ORBIT │ SCORE │       PROVIDER       │
├─────────────

### Selecting a scene

Pick the clearest (lowest cloud cover) scene from the results.


In [5]:
if results:
    scene = min(results, key=lambda r: r.cloud_cover if r.cloud_cover is not None else 100)
    print(f"Selected: {scene.id}")
    print(f"  Date: {scene.datetime}")
    print(f"  Cloud cover: {scene.cloud_cover}%")
    print(f"  Dataset: {scene.properties.get('dataset', 'n/a')}")
else:
    scene = None
    print("No scenes found. Common causes:")
    print("  - M2M access not yet approved for this account")
    print("  - No Landsat coverage matching this AOI/date/cloud filter")
    print("  - Try widening the date range or raising cloud_cover_max")


Selected: LC08_L2SP_193056_20241214_02_T1
  Date: 2024-12-14 10:15:42.019098+00:00
  Cloud cover: 6.91%
  Dataset: n/a


## Part 3 — Download

Downloads use dynamic product resolution (correctly matching the exact
product bundle for this dataset via USGS's `download-options` endpoint),
with resumable retries and a real-time progress bar.


In [ ]:
output_dir = Path("./data/landsat_scene")
output_dir.mkdir(parents=True, exist_ok=True)

if scene is None:
    print("Skipping download — no scene selected above.")
    download_result = None
else:
    options = DownloadOptions(parallel=1, resume=True, on_failure="skip")
    results_dl = client.download([scene], destination=output_dir, options=options)
    download_result = results_dl[0]
    status = "OK" if download_result.success else f"FAILED: {download_result.error}"
    print(f"  {download_result.data_id}: {status}")
    if download_result.success:
        print(f"  Saved to: {download_result.output_path}")


⬇ 1 scene  →  data/landsat_scene              0/1  [00:00]

LC08_L2SP_193056_20241214_02_T1: 0.00B [00:00, ?B/s]

12:56:40 INFO [planetary_computer] Downloading 19 asset(s) for LC08_L2SP_193056_20241214_02_T1: ['qa', 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'qa_pixel', 'qa_radsat', 'qa_aerosol']
12:56:40 INFO [planetary_computer]   Fetching asset 'qa' → LC08_L2SP_193056_20241214_20241218_02_T1_ST_QA.TIF
13:06:03 INFO [planetary_computer]   ✓ LC08_L2SP_193056_20241214_20241218_02_T1_ST_QA.TIF (29.5 MB)
13:06:03 INFO [planetary_computer]   Fetching asset 'red' → LC08_L2SP_193056_20241214_20241218_02_T1_SR_B4.TIF
13:19:12 INFO [planetary_computer]   ✓ LC08_L2SP_193056_20241214_20241218_02_T1_SR_B4.TIF (79.5 MB)
13:19:12 INFO [planetary_computer]   Fetching asset 'blue' → LC08_L2SP_193056_20241214_20241218_02_T1_SR_B2.TIF


## Part 4 — Extracting Bands from the Downloaded Archive

Landsat Collection 2 Level-2 bundles are delivered as a `.tar` archive
containing individual band GeoTIFFs plus the `QA_PIXEL` quality band.
Band naming follows the pattern `<scene_id>_SR_B<N>.TIF` for surface
reflectance bands and `<scene_id>_QA_PIXEL.TIF` for the quality band.

Landsat 8/9 OLI band mapping:

| Band | Wavelength | Common name |
|---|---|---|
| SR_B2 | Blue | Blue |
| SR_B3 | Green | Green |
| SR_B4 | Red | Red |
| SR_B5 | NIR | Near-infrared |
| SR_B6 | SWIR1 | Shortwave infrared 1 |
| SR_B7 | SWIR2 | Shortwave infrared 2 |


In [ ]:
from pygeofetch.processor import LandsatExtractor

extractor = LandsatExtractor()  # mask_clouds=True by default

if download_result is not None and download_result.success:
    landsat_scene = extractor.process_scene(
        download_result, output_dir=output_dir,
        bands=["blue", "green", "red", "nir", "swir1", "swir2"],
    )
    print(f"Sensor: {landsat_scene.sensor}")
    print(f"Available bands: {landsat_scene.available_bands}")
    print(f"Cloud coverage: {landsat_scene.cloud_pct:.1f}%" if landsat_scene.cloud_pct is not None else "No cloud mask applied")
else:
    landsat_scene = None
    print("Skipping extraction — no successful download available.")


In [ ]:
# LandsatExtractor.process_scene() already located, scaled, and
# cloud-masked all requested bands — access them directly:
if landsat_scene is not None:
    red = landsat_scene.get("red")
    nir = landsat_scene.get("nir")
    green = landsat_scene.get("green")
    swir1 = landsat_scene.get("swir1")
    ref_profile = landsat_scene.profile
    cloud_mask = landsat_scene.cloud_mask
    cloud_pct = landsat_scene.cloud_pct
else:
    red = nir = green = swir1 = ref_profile = cloud_mask = cloud_pct = None


## Part 5 — Radiometric Scaling (DN → Surface Reflectance)

Landsat C2L2 delivers scaled integers, not physical reflectance values
directly. The official USGS scale factor and offset must be applied:

$$\text{reflectance} = DN \times 0.0000275 - 0.2$$

(Source: USGS Landsat Collection 2 Level-2 Science Product Guide)


In [ ]:
# Radiometric scaling was already applied by LandsatExtractor.process_scene()
# above — the official USGS scale factor (0.0000275) and offset (-0.2)
# were used automatically, with the correct band mapping for this scene's
# sensor (OLI vs TM/ETM+ use different band numbers for the same wavelength).
if red is not None:
    print(f"Red reflectance range: [{np.nanmin(red):.3f}, {np.nanmax(red):.3f}]")
    print(f"NIR reflectance range: [{np.nanmin(nir):.3f}, {np.nanmax(nir):.3f}]")
else:
    print("Skipping — no bands available.")


## Part 6 — Cloud Masking via QA_PIXEL

Collection 2's `QA_PIXEL` band encodes cloud/shadow/snow/water flags as
individual **bits** within each 16-bit pixel value — not simple class
integers. The relevant bits (per the official Landsat 8-9 C2 Level-2
Science Product Guide, Table 6-2):

| Bit | Flag |
|---|---|
| 1 | Dilated Cloud |
| 3 | Cloud |
| 4 | Cloud Shadow |

We mask out any pixel where bit 1, 3, or 4 is set.


In [ ]:
# Cloud/shadow masking was already applied by LandsatExtractor.process_scene()
# above via QA_PIXEL bit-flag decoding (bits 1, 3, 4 — Dilated Cloud, Cloud,
# Cloud Shadow — per the official Landsat 8-9 C2 Level-2 Science Product Guide).
if cloud_pct is not None:
    print(f"Cloud/shadow-flagged pixels: {cloud_pct:.1f}% of the scene")
    print(f"Valid (non-cloud, non-fill) pixels: {100*np.mean(~np.isnan(red)):.1f}%")
else:
    print("No cloud mask available for this scene.")

red_masked, nir_masked = red, nir  # already masked by LandsatExtractor


## Part 7 — Spectral Indices

With cloud-masked, scaled reflectance in hand, compute standard indices
via `pygeofetch.processor.SpectralIndex`.


In [ ]:
from pygeofetch.processor import SpectralIndex

si = SpectralIndex(prefer_spyndex=False)  # built-in formulae, no extra dependency

if red_masked is not None and nir_masked is not None:
    ndvi = si.compute("NDVI", RED=red_masked, NIR=nir_masked)
    print(f"NDVI range: [{np.nanmin(ndvi):.3f}, {np.nanmax(ndvi):.3f}]")
    print(f"Mean NDVI: {np.nanmean(ndvi):.3f}")

    if band_paths.get("green"):
        green, _ = load_scaled_reflectance(band_paths["green"])
        green_masked = np.where(cloud_mask, np.nan, green) if cloud_mask is not None else green
        ndwi = si.compute("NDWI", GREEN=green_masked, NIR=nir_masked)
        print(f"NDWI range: [{np.nanmin(ndwi):.3f}, {np.nanmax(ndwi):.3f}]")

    if band_paths.get("swir1"):
        swir1, _ = load_scaled_reflectance(band_paths["swir1"])
        swir1_masked = np.where(cloud_mask, np.nan, swir1) if cloud_mask is not None else swir1
        ndbi = si.compute("NDBI", SWIR1=swir1_masked, NIR=nir_masked)
        print(f"NDBI range: [{np.nanmin(ndbi):.3f}, {np.nanmax(ndbi):.3f}]")
else:
    print("Skipping — required bands not available.")


## Part 8 — Visualization


In [ ]:
import matplotlib.pyplot as plt

if 'ndvi' in dir() and ndvi is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    im0 = axes[0].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
    axes[0].set_title(f"NDVI — {scene.datetime if scene else ''}")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, label="NDVI")

    if cloud_mask is not None:
        axes[1].imshow(cloud_mask, cmap="gray")
        axes[1].set_title(f"Cloud/shadow mask ({cloud_pct:.1f}% masked)")
    else:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, "No cloud mask available", ha="center")

    plt.tight_layout()
    plt.savefig("landsat_ndvi_preview.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("\nSaved: landsat_ndvi_preview.png")


## Part 9 — Save Outputs


In [ ]:
from pygeofetch.processing.base import _safe_write_band

if 'ndvi' in dir() and ndvi is not None:
    out_dir = output_dir / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)

    _safe_write_band(ndvi, ref_profile, out_dir / "ndvi.tif")
    print(f"Saved: {out_dir / 'ndvi.tif'}")

    if cloud_mask is not None:
        _safe_write_band(
            cloud_mask.astype(np.float32), ref_profile, out_dir / "cloud_mask.tif"
        )
        print(f"Saved: {out_dir / 'cloud_mask.tif'}")


## Summary

You've now walked through the complete PyGeoFetch USGS Landsat chain:

1. ✅ **Authentication** via M2M Application Token (not password)
2. ✅ **Search** Landsat Collection 2 Level-2 by AOI, date, cloud cover
3. ✅ **Download** with dynamic product resolution and resumable retries
4. ✅ **Extraction** of individual bands from the `.tar` bundle
5. ✅ **Radiometric scaling** — DN to physical surface reflectance
6. ✅ **Cloud masking** via QA_PIXEL bit-flag decoding
7. ✅ **Spectral indices** — NDVI, NDWI, NDBI
8. ✅ **Visualization and export**

**Next:** see `13_usgs_ghana_deforestation_monitoring.ipynb` for a
complete real-world project applying this chain to land degradation
monitoring in Ghana.

### References

- USGS. *M2M Application Token Documentation.*
  https://www.usgs.gov/media/files/m2m-application-token-documentation
- USGS. *Landsat 8-9 Collection 2 Level-2 Science Product Guide* (LSDS-1619).
- USGS. *How do I use a scale factor with Landsat Level-2 science products?*
  https://www.usgs.gov/faqs/how-do-i-use-a-scale-factor-landsat-level-2-science-products
